# A character-level language model, and sampling from it

p(token | past tokens), trained in two minutes, then called in a loop to generate text — including the surgery that generation requires and the experiment that shows why bidirectional would break it.

**Runs on:** CPU — about 5 minutes (GPU: 2 minutes) &nbsp;·&nbsp; **Slides:** [Chapter 15 — Language Models and the Transformer](../../../course-web-slides/ch15/index.html) &nbsp;·&nbsp; **Section:** 01 — The language model

---

## The corpus

In [ ]:
import keras
import tensorflow as tf
import numpy as np

filename = keras.utils.get_file(
    origin=("https://storage.googleapis.com/download.tensorflow.org/"
            "data/shakespeare.txt"))
shakespeare = open(filename, "r").read()

print(f"{len(shakespeare):,} characters")
print(shakespeare[:250])

## Features and labels differ by one character

In [ ]:
sequence_length = 100

def split_input(inp, sequence_length):
    for i in range(0, len(inp), sequence_length):
        yield inp[i:i + sequence_length]

features = list(split_input(shakespeare[:-1], sequence_length))
labels = list(split_input(shakespeare[1:], sequence_length))
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

x, y = next(dataset.as_numpy_iterator())
print(repr(x[:50]))
print(repr(y[:50]))

`shakespeare[:-1]` against `shakespeare[1:]`. **That single line is the entire supervision signal** — and one 100-character input produces 100 supervised predictions, not one.

## A 67-character vocabulary

In [ ]:
from keras import layers

tokenizer = layers.TextVectorization(
    standardize=None, split="character",
    output_sequence_length=sequence_length)
tokenizer.adapt(dataset.map(lambda text, labels: text))

vocabulary_size = tokenizer.vocabulary_size()
print(f"vocabulary: {vocabulary_size} characters")
print(tokenizer.get_vocabulary()[:20])

dataset = dataset.map(
    lambda f, l: (tokenizer(f), tokenizer(l)), num_parallel_calls=8)
training_data = dataset.shuffle(10_000).batch(64).cache()

## The model

In [ ]:
embedding_dim, hidden_dim = 256, 1024

inputs = layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")
x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = layers.GRU(hidden_dim, return_sequences=True)(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.summary()

`return_sequences=True` is essential — a prediction at **every** position, not just the last. Note also that ~98% of the parameters are in the GRU; that balance shifts dramatically in the Transformer models later in this chapter.

## Training

In [ ]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["sparse_categorical_accuracy"])
model.fit(training_data, epochs=20, verbose=2)

A batch of 64 sequences of 100 characters is **6,400 individual classifications**. Around 70% next-character accuracy after 20 epochs.

## Surgery for generation

In [ ]:
inputs = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")

x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state = layers.GRU(hidden_dim, return_state=True)(
    x, initial_state=input_state)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)

generation_model = keras.Model(inputs=(inputs, input_state),
                               outputs=(outputs, output_state))
generation_model.set_weights(model.get_weights())
print("same weights, different interface")

One token in, and the GRU state promoted from an internal detail to a **named input and output**. Same computational structure, so the weights transfer directly.

## Priming, then sampling

In [ ]:
tokens = tokenizer.get_vocabulary()
char_to_id = dict(zip(tokens, range(vocabulary_size)))
id_to_char = dict(zip(range(vocabulary_size), tokens))

prompt = "\nKING RICHARD III:\n"
input_ids = [char_to_id[c] for c in prompt]

state = keras.ops.zeros(shape=(1, hidden_dim))
for token_id in input_ids:
    inp = keras.ops.expand_dims([token_id], axis=0)
    predictions, state = generation_model.predict((inp, state), verbose=0)

generated_ids = []
for i in range(250):
    next_char = int(np.argmax(predictions, axis=-1)[0])
    generated_ids.append(next_char)
    inp = keras.ops.expand_dims([next_char], axis=0)
    predictions, state = generation_model.predict((inp, state), verbose=0)

print(prompt + "".join(id_to_char[t] for t in generated_ids))

Correctly spelled words, speaker names in capitals followed by a colon, blank lines between speeches, verse-length lines. **All of it from a next-character objective**, none of it encoded.

## The diagnostic: replace GRU with Bidirectional(GRU)

In [ ]:
bi = keras.Sequential([
    layers.Input(shape=(sequence_length,), dtype="int"),
    layers.Embedding(vocabulary_size, embedding_dim),
    layers.Bidirectional(layers.GRU(256, return_sequences=True)),
    layers.Dense(vocabulary_size, activation="softmax"),
])
bi.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
           metrics=["sparse_categorical_accuracy"])
h = bi.fit(training_data.take(100), epochs=3, verbose=2)

print(f"\nbidirectional training accuracy after 3 epochs: "
      f"{h.history['sparse_categorical_accuracy'][-1]:.4f}")
print("compare: the causal model reached ~0.70 after 20 epochs")

> ⚠️ **Accuracy shoots above 99% almost immediately, and generation is dead.** The backward pass hands the model the next character as a feature — it is reading the label off its own input.

**A training metric that suddenly looks too good is the most reliable signal of leakage there is.** This is chapter 5's lesson, and chapter 15's causal mask exists to prevent exactly it.

## Temperature, briefly

In [ ]:
def sample(prompt, temperature=1.0, length=200):
    ids = [char_to_id[c] for c in prompt]
    st = keras.ops.zeros(shape=(1, hidden_dim))
    for t in ids:
        p, st = generation_model.predict(
            (keras.ops.expand_dims([t], axis=0), st), verbose=0)
    out = []
    for _ in range(length):
        logits = np.log(np.maximum(p[0], 1e-9)) / temperature
        probs = np.exp(logits); probs /= probs.sum()
        nxt = int(np.random.choice(len(probs), p=probs))
        out.append(nxt)
        p, st = generation_model.predict(
            (keras.ops.expand_dims([nxt], axis=0), st), verbose=0)
    return "".join(id_to_char[i] for i in out)

for t in [0.2, 0.7, 1.3]:
    print(f"\n--- temperature {t} ---")
    print(sample(prompt, temperature=t, length=180))

Low temperature repeats; high temperature loses the spelling. **Chapter 16 makes this a first-class control** and adds top-K beside it.

---

## What to take away

- A language model is p(token | past tokens); labels are the features shifted by one.
- Generation needs an inference model with the recurrent state made explicit.
- **Bidirectional breaks it** — 99% training accuracy, dead generation. Leakage looks like success.
- There is logic in the generation loop that has no counterpart in training.